# 02 · Traduction de la parole tunisienne (Speech-to-Text Translation)

**Projet commun GenAI / NLP · corpus TEDxTN**

**Objectifs**
1. Exploiter les paires *audio tunisien → traduction anglaise* de TEDxTN.
2. Mesurer des **baselines** zero-shot (Whisper, SeamlessM4T).
3. Entraîner une approche **directe (end-to-end)** : Whisper fine-tuné en tâche `translate`.
4. La comparer à un **pipeline en deux étapes** : ASR tunisien (notebook 01) → traduction automatique (NLLB, LLM).
5. Évaluer avec **BLEU** (protocole du papier), **chrF++** et **COMET**, avec intervalles de confiance et test
   de significativité, puis analyser les erreurs sur des exemples réels.
6. Produire aussi des traductions vers le **français** et l'**arabe standard** (sans référence : évaluation indirecte).

**Améliorations par rapport au papier**

| Limite | Ce que fait ce notebook |
|---|---|
| pas de comparaison direct vs pipeline | 5 variantes de pipeline + 2 variantes directes, même test |
| AST initialisé depuis Whisper « nu » | E2E initialisé depuis le **modèle ASR fine-tuné** (transfert de tâche) |
| MT sensible aux erreurs ASR | NLLB fine-tuné sur un mélange de transcriptions parfaites et de **sorties ASR bruitées** |
| BLEU seul, une seule référence | chrF++ + COMET + bootstrap apparié + analyse manuelle guidée |
| échec inexpliqué encodeur + NLLB (< 5 BLEU) | expérience de diagnostic avec adaptateur et gel partiel (option) |
| volet GenAI | traduction par LLM (Qwen2.5) avec exemples few-shot, comparée à NLLB |

## 1. Introduction
Traduire de la parole tunisienne cumule les difficultés de l'ASR (dialecte sans orthographe, code-switching)
et de la traduction (peu de textes parallèles tunisien-anglais). Deux stratégies s'opposent :

- **Directe (end-to-end)** : un seul modèle lit l'audio et écrit l'anglais. Pas de propagation d'erreurs
  d'une transcription intermédiaire, mais il faut apprendre la reconnaissance et la traduction avec peu de données.
- **Pipeline (cascade)** : ASR puis traduction de texte. Chaque module est plus simple et réutilise des modèles
  pré-entraînés puissants (NLLB connaît le tunisien, `aeb_Arab`), mais les erreurs de l'ASR se propagent.

Le papier ne rapporte que l'approche directe (Whisper-large-v3 fine-tuné : 25,68 BLEU). Nous comparons les deux.

## 2. Installation et configuration

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/roudayn-kouka/tedxtn-tunisian-speech.git"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("TEDXTN_WORKDIR", "/content/drive/MyDrive/tedxtn")
    project = Path("/content/drive/MyDrive/tedxtn-tunisian-speech")
    if not project.exists():
        project = Path("/content/tedxtn-tunisian-speech")
        if not project.exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, str(project)], check=True)
    os.chdir(project)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import json, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

from common import data, metrics, text
from common.config import (BASE_MODELS, DATA_DIR, MODELS_DIR, PAPER_AST_RESULTS, RESULTS_DIR, SEED, TARGET_LANGUAGES,
                           TrainConfig)

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
print("GPU disponible :", HAS_GPU)

In [ ]:
from common.config import REUSE_PREDS, SMOKE, env_flag, gpu_memory_gb, trained
VRAM = gpu_memory_gb()

# Chaque interrupteur peut être forcé sans modifier le notebook : TEDXTN_RUN_<NOM>=0/1.
RUN_ZERO_SHOT = env_flag("ZERO_SHOT", HAS_GPU)        # B0 Whisper zero-shot
RUN_SEAMLESS = env_flag("SEAMLESS", HAS_GPU and VRAM >= 12)   # B1 SeamlessM4T-v2-large (2,3 B paramètres)
RUN_E2E = env_flag("E2E", HAS_GPU)                    # E2E-1 (depuis Whisper), E2E-2 (depuis l'ASR fine-tuné)
RUN_CASCADE = env_flag("CASCADE", HAS_GPU)            # C0-C1 (NLLB zero-shot)
RUN_NLLB_FT = env_flag("NLLB_FT", HAS_GPU)            # C2
RUN_ASR_NOISE = env_flag("ASR_NOISE", False)          # C3 : nécessite les sorties ASR sur le train (≈ 30 min d'inférence en plus)
RUN_LLM = env_flag("LLM", HAS_GPU)                    # C4 (GenAI)
RUN_COMET = env_flag("COMET", HAS_GPU)
RUN_ENC_DEC_DIAGNOSTIC = env_flag("ENC_DEC_DIAGNOSTIC", False)   # option coûteuse (w2v-BERT + mBART)
RUN_OTHER_TARGETS = env_flag("OTHER_TARGETS", HAS_GPU)           # français et arabe standard

E2E_MODEL = "whisper-small"      # "whisper-large-v3" (LoRA) sur A100/L4
NLLB = BASE_MODELS["nllb-600m"]  # "nllb-1.3b" si la mémoire le permet
# NLLB-600M en fine-tuning complet ne tient pas sous 12 Go (poids + gradients + Adam) : LoRA (r=16 sur
# q/k/v/out_proj + fc1/fc2), poids en fp32 et précision mixte fp16. Écart au fine-tuning complet, signalé dans RESULTATS.md.
NLLB_LORA = env_flag("NLLB_LORA", VRAM < 12)
LLM_MODEL = BASE_MODELS["llm"] if VRAM >= 20 else BASE_MODELS["llm-small"]
GEN_BS = 16 if VRAM >= 12 else 4
print(f"VRAM {VRAM:.1f} Go | SMOKE={SMOKE} | LLM {LLM_MODEL} | batch inférence {GEN_BS}")

## 3. Données
On réutilise le corpus préparé au notebook 01 (segments audio découpés). On ne garde que les segments qui
ont une traduction.

In [ ]:
df = data.load_corpus(download=os.environ.get("TEDXTN_SKIP_DOWNLOAD") != "1")
df = data.cut_segments(df)
df = df[df.translation.str.strip() != ""].reset_index(drop=True)
train_df, rejected = data.clean_for_training(df[df.split == "train"])
dev_df = df[df.split == "dev"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
if SMOKE:
    # Essai rapide (TEDXTN_SMOKE=1) : uniquement des talks de TRAIN déjà découpés. Deux talks de train tenus à
    # part jouent le rôle de dev et de test : le dev et le test officiels ne sont pas utilisés.
    _tr = df[df.split == "train"]
    _talks = sorted(_tr.talk.unique())[:5]   # 5 premiers talks (ordre alphabétique) : même sélection d'un notebook à l'autre
    assert len(_talks) == 5, "smoke test : il faut au moins 5 talks de train découpés"
    dev_df = _tr[_tr.talk == _talks[3]].head(50).assign(split="dev").reset_index(drop=True)
    test_df = _tr[_tr.talk == _talks[4]].head(50).assign(split="test").reset_index(drop=True)
    train_df = train_df[train_df.talk.isin(_talks[:3])]
    train_df = train_df.sample(min(200, len(train_df)), random_state=SEED).reset_index(drop=True)
    print("SMOKE TEST :", {s: len(x) for s, x in [("train", train_df), ("pseudo-dev", dev_df), ("pseudo-test", test_df)]})
print({s: len(x) for s, x in [("train", train_df), ("dev", dev_df), ("test", test_df)]})

In [ ]:
st = df.groupby("split").agg(segments=("seg_id", "size"), mots_tun=("n_words", "sum"), mots_en=("n_words_en", "sum"),
                             mots_tun_moy=("n_words", "mean"), mots_en_moy=("n_words_en", "mean")).reindex(["train", "dev", "test"])
st["ratio en/tun"] = st.mots_en / st.mots_tun
display(st.round(2))
fig, ax = plt.subplots(figsize=(6, 3.5))
tr_ = df[df.split == "train"]; s = tr_.sample(min(3000, len(tr_)), random_state=SEED)
ax.scatter(s.n_words, s.n_words_en, s=4, alpha=.3, color="#c2410c"); ax.plot([0, 80], [0, 80], "--", color="#64748b")
ax.set_xlabel("mots tunisiens"); ax.set_ylabel("mots anglais"); ax.set_title("Longueur source vs cible (train)")
plt.tight_layout(); plt.show()

In [ ]:
for r in df[(df.split == "train") & (df.seg_type == "MIXED")].sample(3, random_state=3).itertuples():
    print(f"TUN : {r.transcription}\nEN  : {r.translation}\n")

**Observation.** La traduction est plus courte que la source en nombre de mots (articles et pronoms clitiques
de l'arabe, hésitations non traduites). Les passages en français doivent eux aussi être traduits en anglais
(consigne n° 7 des annotateurs) : un système qui « recopie » le français sera pénalisé.

## 4. Méthodologie

| Id | Système | Type | Entraînement |
|---|---|---|---|
| B0 | Whisper zero-shot (`translate`) | direct | aucun |
| B1 | SeamlessM4T-v2 (S2TT, source arabe) | direct | aucun |
| E2E-1 | Whisper fine-tuné `translate` | direct | audio → anglais |
| E2E-2 | idem, initialisé depuis l'**ASR fine-tuné** | direct | ASR puis audio → anglais |
| C0 | **transcription de référence** → NLLB | pipeline oracle | aucun (borne haute du pipeline) |
| C1 | meilleur ASR → NLLB zero-shot (`aeb_Arab`) | pipeline | aucun |
| C2 | meilleur ASR → NLLB fine-tuné | pipeline | texte tunisien → anglais |
| C3 | meilleur ASR → NLLB fine-tuné avec sorties ASR bruitées | pipeline | idem + bruit ASR |
| C4 | meilleur ASR (+ correction LLM) → LLM few-shot | pipeline GenAI | aucun |

**Métriques.** BLEU *true-case sans ponctuation* (protocole du papier, sacreBLEU), BLEU sacreBLEU standard,
chrF++ (plus fiable que BLEU pour une morphologie riche et une seule référence) et COMET-22 (corrélé au jugement
humain). IC à 95 % par bootstrap ; test apparié entre le meilleur direct et le meilleur pipeline.
Tous les choix sont faits sur le dev.

In [ ]:
from common import asr as A, translate as T

def wavs(frame):
    return [data.load_audio(p) for p in frame.path]

def best_asr_name():
    """Système ASR retenu par le notebook 01 (meilleur sur le dev)."""
    src = MODELS_DIR / "asr_best" / "SOURCE.txt"
    return src.read_text().strip() if src.exists() else None

## 5. Baselines zero-shot

In [ ]:
if RUN_ZERO_SHOT:
    test_w = wavs(test_df)
    for name in ["whisper-small", "whisper-large-v3"]:
        if data.have_preds("ast", f"B0-{name}-zeroshot", ["test"]):
            print(name, ": prédictions existantes relues"); continue
        proc, model = A.load_whisper(BASE_MODELS[name], task="translate", for_training=False)
        model = model.half()
        hyps = A.whisper_generate(model, proc, test_w, task="translate", batch_size=GEN_BS, num_beams=4)
        data.save_preds("ast", f"B0-{name}-zeroshot", "test", test_df.seg_id, hyps, test_df.translation)
        print(name, metrics.translation_scores(hyps, test_df.translation))
        del model; torch.cuda.empty_cache()
if RUN_SEAMLESS:
    proc, model = T.load_seamless(BASE_MODELS["seamless-m4t-v2"])
    hyps = T.seamless_s2tt(proc, model, wavs(test_df), "eng")
    data.save_preds("ast", "B1-seamless-m4t-v2-zeroshot", "test", test_df.seg_id, hyps, test_df.translation)
    del model; torch.cuda.empty_cache()

## 6. Approche directe (end-to-end)
Whisper en tâche `translate` : le décodeur génère directement l'anglais à partir de l'audio tunisien.
**E2E-2** part des poids du modèle ASR fine-tuné du notebook 01 : l'encodeur sait déjà « entendre » le
tunisien, il ne reste qu'à apprendre à traduire (transfert ASR → AST, classique en traduction de la parole).

In [ ]:
from common.augment import Augmenter

def run_e2e(init, tag, use_lora=False):
    c = TrainConfig(use_lora=use_lora, max_epochs=8)
    out = MODELS_DIR / f"ast_{tag}"
    if trained(out):   # passe finale : pas de réentraînement
        print("modèle final trouvé, entraînement sauté :", out / "final"); return out / "final"
    proc, model = A.load_whisper(init, task="translate", use_lora=use_lora, cfg=c)
    sub = dev_df.sample(min(200, len(dev_df)), random_state=SEED)       # suivi BLEU / chrF++ sur 200 segments du dev
    cb = A.GenerationWERCallback(lambda m, w: A.whisper_generate(m, proc, w, task="translate", batch_size=GEN_BS),
                                 wavs(sub), sub.translation.tolist(), kind="ast")
    trainer = A.make_trainer(model, A.training_args(out, c, seq2seq=True),
                             A.SpeechDataset(train_df, text_col="translation", augmenter=Augmenter(seed=SEED)),
                             A.SpeechDataset(dev_df, text_col="translation"),
                             A.WhisperCollator(proc, task="translate"), proc, c, [cb.as_callback()])
    last = max(out.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1])) if out.exists() else None
    trainer.train(resume_from_checkpoint=str(last) if last else None)
    pd.DataFrame(trainer.state.log_history).to_csv(RESULTS_DIR / f"train_log_ast_{tag}.csv", index=False)
    pd.DataFrame(cb.history).to_csv(RESULTS_DIR / f"dev_curve_ast_{tag}.csv", index=False)
    return A.save_for_inference(trainer.model, proc, out / "final")

def predict_e2e(model_dir, system):
    if data.have_preds("ast", system):
        print(system, ": prédictions existantes relues"); return
    proc, model = A.load_whisper(str(model_dir), task="translate", for_training=False)
    model = model.half()
    for split, frame in [("dev", dev_df), ("test", test_df)]:
        h = A.whisper_generate(model, proc, wavs(frame), task="translate", batch_size=GEN_BS, num_beams=4)
        data.save_preds("ast", system, split, frame.seg_id, h, frame.translation)
    del model; torch.cuda.empty_cache()

if RUN_E2E:
    lora = E2E_MODEL != "whisper-small"
    predict_e2e(run_e2e(BASE_MODELS[E2E_MODEL], f"{E2E_MODEL}-e2e", lora), f"E2E-1-{E2E_MODEL}")
    # E2E-2 : initialisé depuis le Whisper ASR fine-tuné du notebook 01 (même architecture), que le meilleur ASR
    # du dev soit ce Whisper ou le w2v-BERT CTC (le transfert encodeur-décodeur demande un Whisper).
    asr_whisper = MODELS_DIR / f"asr_{E2E_MODEL}" / "final"
    if (asr_whisper / "config.json").exists():
        predict_e2e(run_e2e(str(asr_whisper), f"{E2E_MODEL}-e2e-from-asr", lora), f"E2E-2-{E2E_MODEL}-from-asr")
    else:
        print("E2E-2 ignoré : pas de Whisper ASR fine-tuné (notebook 01, section 6).")

## 7. Pipeline en deux étapes
**Étape 1** : transcriptions du meilleur ASR (fichiers `results/preds/asr__*`, produits au notebook 01).
**Étape 2** : traduction du texte.

NLLB-200 est utilisé avec la langue source `aeb_Arab` (arabe tunisien). On l'évalue d'abord sur les
**transcriptions de référence** (C0) : c'est la borne haute du pipeline, qui isole la qualité de la MT.

In [ ]:
asr_name = best_asr_name()
asr_test = data.load_preds("asr", "test").get(asr_name) if asr_name else None
asr_dev = data.load_preds("asr", "dev").get(asr_name) if asr_name else None
print("ASR utilisé pour le pipeline :", asr_name, "" if asr_test is not None else "(introuvable : exécuter le notebook 01)")

def asr_text(frame, preds):
    m = dict(zip(preds.seg_id, preds.hyp))
    return [m.get(s, "") for s in frame.seg_id]

if RUN_CASCADE and not (data.have_preds("ast", "C0-oracle-ref->nllb", ["test"]) and data.have_preds("ast", "C1-asr->nllb-zeroshot")):
    tok, mt = T.load_nllb(NLLB)
    h = T.nllb_translate(tok, mt, test_df.transcription.tolist(), "eng_Latn")
    data.save_preds("ast", "C0-oracle-ref->nllb", "test", test_df.seg_id, h, test_df.translation)
    if asr_test is not None:
        for split, frame, preds in [("dev", dev_df, asr_dev), ("test", test_df, asr_test)]:
            if preds is None:
                continue
            src = asr_text(frame, preds)
            h = T.nllb_translate(tok, mt, src, "eng_Latn")
            data.save_preds("ast", "C1-asr->nllb-zeroshot", split, frame.seg_id, h, frame.translation, source=src)
    del mt; torch.cuda.empty_cache()

### 7.1 Fine-tuning de NLLB sur les paires TEDxTN (C2) et robustesse aux erreurs ASR (C3)
C2 apprend sur (transcription de référence → traduction). C3 remplace la moitié des sources par la sortie
de l'ASR sur le train : le modèle apprend à traduire des entrées imparfaites, comme celles qu'il recevra.

In [ ]:
def finetune_nllb(train_frame, tag):
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    out = MODELS_DIR / f"mt_{tag}"
    if trained(out):   # passe finale : pas de réentraînement
        print("modèle final trouvé, entraînement sauté :", out / "final"); return out / "final"
    tok = AutoTokenizer.from_pretrained(NLLB, src_lang="aeb_Arab", tgt_lang="eng_Latn")
    model = AutoModelForSeq2SeqLM.from_pretrained(NLLB)          # poids en fp32 (fp16 = précision mixte du Trainer)
    c = TrainConfig(learning_rate=5e-5, lora_learning_rate=2e-4, use_lora=NLLB_LORA, batch_size=16, grad_accum=2,
                    max_epochs=6, eval_steps=400, gradient_checkpointing=False, spec_augment=False)
    if NLLB_LORA:
        from peft import LoraConfig, TaskType, get_peft_model
        model = get_peft_model(model, LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, lora_dropout=0.05,
                                                 target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]))
        model.print_trainable_parameters()
    out = MODELS_DIR / f"mt_{tag}"
    args = A.training_args(out, c, seq2seq=True, label_smoothing_factor=0.1)
    trainer = A.make_trainer(model, args, T.TextPairDataset(train_frame, tok), T.TextPairDataset(dev_df, tok),
                             T.Seq2SeqCollator(tok, model), type("P", (), {"feature_extractor": tok})(), c)
    last = max(out.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1])) if out.exists() else None
    trainer.train(resume_from_checkpoint=str(last) if last else None)
    pd.DataFrame(trainer.state.log_history).to_csv(RESULTS_DIR / f"train_log_mt_{tag}.csv", index=False)
    return A.save_for_inference(trainer.model, tok, out / "final")   # LoRA fusionné : chargeable comme un NLLB normal

def cascade_with(mt_dir, system):
    if data.have_preds("ast", system) and data.have_preds("ast", system.replace("asr->", "oracle-ref->"), ["test"]):
        print(system, ": prédictions existantes relues"); return
    tok, mt = T.load_nllb(str(mt_dir))
    for split, frame, preds in [("dev", dev_df, asr_dev), ("test", test_df, asr_test)]:
        if preds is None:
            continue
        src = asr_text(frame, preds)
        data.save_preds("ast", system, split, frame.seg_id, T.nllb_translate(tok, mt, src, "eng_Latn"), frame.translation, source=src)
    h = T.nllb_translate(tok, mt, test_df.transcription.tolist(), "eng_Latn")
    data.save_preds("ast", system.replace("asr->", "oracle-ref->"), "test", test_df.seg_id, h, test_df.translation)
    del mt; torch.cuda.empty_cache()

if RUN_NLLB_FT and asr_test is not None:
    cascade_with(finetune_nllb(train_df, "nllb-ft"), "C2-asr->nllb-ft")
    if RUN_ASR_NOISE:
        best = MODELS_DIR / "asr_best"
        cfgj = json.loads((best / "config.json").read_text())
        if cfgj.get("model_type") == "whisper":
            proc, m = A.load_whisper(str(best), for_training=False); tr_h = A.whisper_generate(m.half(), proc, wavs(train_df), batch_size=GEN_BS)
        else:
            proc, m = A.load_ctc(str(best), None, from_checkpoint=True); tr_h = A.ctc_transcribe(m, proc, wavs(train_df))
        del m; torch.cuda.empty_cache()
        noisy = T.add_asr_noise(train_df.assign(asr_hyp=tr_h), "asr_hyp", p=0.5)
        cascade_with(finetune_nllb(noisy, "nllb-ft-asrnoise"), "C3-asr->nllb-ft-asrnoise")

### 7.2 Volet GenAI : traduction par LLM (C4)
Qwen2.5-7B-Instruct (4 bits) traduit la sortie ASR (corrigée par le LLM si le notebook 01 l'a produite),
avec 5 exemples few-shot tirés du train.

In [ ]:
if RUN_LLM and asr_test is not None:
    from common import llm as L
    tok, llm_model = L.load_llm(LLM_MODEL)
    shots = [(r.transcription, r.translation) for r in train_df[(train_df.n_words.between(6, 20))].sample(5, random_state=SEED).itertuples()]
    for split, frame, base in [("dev", dev_df, asr_dev), ("test", test_df, asr_test)]:
        if base is None:
            continue
        corrected = data.load_preds("asr", split).get(f"{asr_name}+llm")
        src_name = "asr+llm" if corrected is not None else "asr"
        src = asr_text(frame, corrected if corrected is not None else base)
        if data.have_preds("ast", f"C4-{src_name}->llm", [split]):
            continue
        h = L.llm_translate(tok, llm_model, src, "English", shots)
        data.save_preds("ast", f"C4-{src_name}->llm", split, frame.seg_id, h, frame.translation, source=src)
    del llm_model; torch.cuda.empty_cache()

### 7.3 (Option) Diagnostic de l'échec encodeur de parole + décodeur de texte
Le papier obtient < 5 BLEU en branchant les sorties de w2v-BERT sur le décodeur NLLB, sans explication.
Hypothèses testées : (1) séquences encodeur trop longues → adaptateur convolutif de sous-échantillonnage ;
(2) cross-attention aléatoire déstabilisée par un LR trop fort sur l'encodeur → gel des 12 premières couches ;
(3) NLLB n'existe pas en `CausalLM` dans transformers → décodeur mBART-50, de la même famille.
Si ce modèle dépasse nettement 5 BLEU, cela indique que l'échec du papier venait de la configuration
(alignement des longueurs / optimisation) plutôt que de l'architecture.

In [ ]:
if RUN_ENC_DEC_DIAGNOSTIC:
    fe, dtok, ed = T.build_speech_encoder_decoder()
    c = TrainConfig(batch_size=8, grad_accum=4, learning_rate=3e-5, max_epochs=10, spec_augment=False)
    out = MODELS_DIR / "ast_w2vbert-mbart"
    trainer = A.make_trainer(ed, A.training_args(out, c, seq2seq=True),
                             A.SpeechDataset(train_df, text_col="translation"), A.SpeechDataset(dev_df, text_col="translation"),
                             T.SpeechTextCollator(fe, dtok), type("P", (), {"feature_extractor": fe})(), c)
    trainer.train()
    ed.eval(); hyps = []
    for i in range(0, len(test_df), 8):
        f = fe(wavs(test_df.iloc[i:i + 8]), sampling_rate=16000, return_tensors="pt", padding=True)
        g = ed.generate(inputs=f["input_features"].to(ed.device), attention_mask=f["attention_mask"].to(ed.device), num_beams=4, max_new_tokens=200)
        hyps += dtok.batch_decode(g, skip_special_tokens=True)
    data.save_preds("ast", "D-w2vbert+mbart-adapter", "test", test_df.seg_id, hyps, test_df.translation)

## 8. Résultats

In [ ]:
comet_model = None
if RUN_COMET:
    # installé ici seulement : unbabel-comet impose ses propres versions de dépendances.
    # setuptools < 81 : torchmetrics 0.10 (dépendance de COMET) importe pkg_resources, retiré des setuptools récents.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unbabel-comet>=2.2", "setuptools<81"], check=True)
    try:
        from comet import download_model, load_from_checkpoint
        comet_model = load_from_checkpoint(download_model(BASE_MODELS["comet"]))
    except Exception as e:   # COMET indisponible : BLEU et chrF++ restent calculés, la colonne COMET reste vide
        print("COMET indisponible :", type(e).__name__, e)

def system_type(name):
    return "oracle" if "oracle" in name else "direct" if name[:1] in "BE" or name.startswith("D-") else "pipeline"

def evaluate_ast(split, frame, with_comet):
    src_by_seg = dict(zip(frame.seg_id, frame.transcription))
    rows, seg = [], {}
    for name, p in data.load_preds("ast", split).items():
        hyps, refs = p.hyp.astype(str).tolist(), p.ref.astype(str).tolist()
        sc = metrics.translation_scores(hyps, refs, [src_by_seg.get(s, "") for s in p.seg_id], comet_model if with_comet else None)
        _, lo, hi = metrics.bootstrap_ci(metrics.bleu_fn(), hyps, refs, n_samples=500)
        seg[name] = p.assign(chrf=[metrics.sentence_chrf(h, r) for h, r in zip(hyps, refs)])
        rows.append({"système": name, "type": system_type(name), "BLEU": sc["BLEU"], "IC95 BLEU": f"[{lo:.1f} ; {hi:.1f}]",
                     "BLEU sacre": sc["BLEU_sacre"], "chrF++": sc["chrF++"], "COMET": sc.get("COMET", np.nan),
                     "segments": len(p)})
    res = pd.DataFrame(rows).sort_values("chrF++", ascending=False).round(2) if rows else pd.DataFrame()
    return res, seg

# Tous les choix (meilleur direct, meilleur pipeline, modèles exportés) se font sur le DEV ; le test est rapporté.
ast_dev, seg_scores_dev = evaluate_ast("dev", dev_df, with_comet=False)
ast_res, seg_scores = evaluate_ast("test", test_df, with_comet=True)
src_by_seg = dict(zip(test_df.seg_id, test_df.transcription))   # sources tunisiennes du test (analyse d'erreurs, §9)
if len(ast_dev):
    ast_dev.to_csv(RESULTS_DIR / "ast_results_dev.csv", index=False)
    print("DEV (sert aux choix) :"); display(ast_dev)

def best_on_dev(kind, prefix=None):
    """Meilleur système d'un type sur le dev (chrF++), parmi ceux qui ont aussi une sortie de test."""
    if not len(ast_dev):
        return None
    d = ast_dev[(ast_dev.type == kind) & ast_dev["système"].isin(set(ast_res.get("système", [])))]
    if prefix:
        d = d[d["système"].str.startswith(prefix)]
    return d.iloc[0]["système"] if len(d) else None

if len(ast_res):
    ast_res["retenu (dev)"] = ast_res["système"].isin({best_on_dev("direct"), best_on_dev("pipeline")})
    ast_res.to_csv(RESULTS_DIR / "ast_results.csv", index=False)
    print("TEST :"); display(ast_res)
    display(pd.DataFrame([{"système (papier)": k, "BLEU": v} for k, v in PAPER_AST_RESULTS.items()]))

In [ ]:
if len(ast_res):
    r = ast_res[ast_res.type != "oracle"]
    fig, ax = plt.subplots(figsize=(10, .45 * len(r) + 1))
    colors = r.type.map({"direct": "#c2410c", "pipeline": "#0f766e"})
    ax.barh(r["système"], r["chrF++"], color=colors); ax.invert_yaxis(); ax.set_xlabel("chrF++ (test)")
    for y, (b, c) in enumerate(zip(r["BLEU"], r["chrF++"])):
        ax.text(c, y, f"  BLEU {b:.1f}", va="center", fontsize=8)
    ax.set_title("Direct (orange) vs pipeline (vert)")
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "02_ast_comparison.png", bbox_inches="tight"); plt.show()

### 8.1 Le meilleur direct est-il significativement différent du meilleur pipeline ?

In [ ]:
if len(ast_res):
    # meilleur direct et meilleur pipeline choisis sur le dev, comparés sur le test
    a_name, b_name = best_on_dev("direct"), best_on_dev("pipeline")
    if a_name and b_name:
        A_, B_ = seg_scores[a_name], seg_scores[b_name].set_index("seg_id").loc[seg_scores[a_name].seg_id].reset_index()
        for label, fn in [("BLEU", metrics.bleu_fn()), ("chrF++", metrics.chrf_fn())]:
            res = metrics.paired_bootstrap(fn, A_.hyp.tolist(), B_.hyp.tolist(), A_.ref.tolist(), n_samples=1000)
            print(f"{label} : {b_name} − {a_name} = {res['delta_moyen']:+.2f} (IC95 {res['IC95'][0]:+.2f} ; {res['IC95'][1]:+.2f}), "
                  f"p = {res['p_value']:.3f}")

## 9. Analyse des erreurs
### 9.1 Propagation des erreurs dans le pipeline
Pour chaque segment, on compare le WER de l'ASR à la qualité de la traduction en cascade.
Une forte corrélation négative confirme que l'étape 1 est le goulot d'étranglement.

In [ ]:
casc = best_on_dev("pipeline") if len(ast_res) else None   # pipeline retenu sur le dev
if casc and asr_test is not None:
    S = seg_scores[casc].merge(asr_test[["seg_id", "hyp", "ref"]].rename(columns={"hyp": "asr", "ref": "tun"}), on="seg_id")
    S["wer"] = [metrics.segment_wer(r, h) for r, h in zip(S.tun, S.asr)]
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.scatter(S.wer.clip(upper=150), S.chrf, s=6, alpha=.35, color="#0f766e")
    ax.set_xlabel("WER ASR du segment (%)"); ax.set_ylabel("chrF++ de la traduction"); ax.set_title(casc)
    plt.tight_layout(); plt.savefig(RESULTS_DIR / "02_error_propagation.png", bbox_inches="tight"); plt.show()
    print("Corrélation de Spearman WER / chrF :", round(S[["wer", "chrf"]].corr("spearman").iloc[0, 1], 3))
    display(S.groupby(pd.cut(S.wer, [-1, 10, 30, 60, 1000])).chrf.agg(["mean", "size"]).round(1))

### 9.2 Qualité selon le type de segment (TUN / MIXED / FOR)

In [ ]:
if len(ast_res):
    tt = []
    for name in ast_res[ast_res.type != "oracle"]["système"].head(6):
        Sx = seg_scores[name].merge(test_df[["seg_id", "seg_type"]], on="seg_id")
        for t, g in Sx.groupby("seg_type"):
            tt.append({"système": name, "type": t, "chrF++": metrics.chrf_fn()(g.hyp.tolist(), g.ref.tolist()), "segments": len(g)})
    display(pd.DataFrame(tt).pivot(index="système", columns="type", values="chrF++").round(1))

### 9.3 Catégories d'erreurs
Pré-classement heuristique (omission, ajout/hallucination, répétition, mots non traduits, français recopié,
contresens), puis **fiche d'annotation manuelle** de 60 segments à compléter par l'équipe
(`results/02_manual_annotation.csv`) pour valider ces catégories.

In [ ]:
if len(ast_res):
    best_direct = next((n for n in ast_res["système"] if ast_res.set_index("système").loc[n, "type"] == "direct"), None)
    best_casc = next((n for n in ast_res["système"] if ast_res.set_index("système").loc[n, "type"] == "pipeline"), None)
    cats = {}
    for name in [best_direct, best_casc]:
        if not name:
            continue
        Sx = seg_scores[name]
        flags = [f for s, h, r in zip(Sx.seg_id, Sx.hyp, Sx.ref) for f in metrics.translation_error_flags(src_by_seg.get(s, ""), h, r)]
        cats[name] = pd.Series(flags).value_counts(normalize=True).mul(100).round(1)
    display(pd.DataFrame(cats).fillna(0))
    if best_direct and best_casc:
        M = seg_scores[best_direct][["seg_id", "hyp", "ref"]].rename(columns={"hyp": "direct"}).merge(
            seg_scores[best_casc][["seg_id", "hyp"]].rename(columns={"hyp": "pipeline"}), on="seg_id").sample(min(60, len(test_df)), random_state=SEED)
        M.insert(1, "source_tun", M.seg_id.map(src_by_seg))
        for col in ["catégorie_direct", "catégorie_pipeline", "meilleure (direct/pipeline/égal)", "commentaire"]:
            M[col] = ""
        M.to_csv(RESULTS_DIR / "02_manual_annotation.csv", index=False)

### 9.4 Exemples corrects et incorrects

In [ ]:
if len(ast_res) and best_casc:
    Sx = seg_scores[best_casc].merge(test_df[["seg_id", "transcription"]], on="seg_id")
    Sx = Sx[Sx.ref.str.split().str.len() >= 6]
    pick = pd.concat([Sx.nlargest(4, "chrf").assign(verdict="✅ correcte"), Sx.nsmallest(4, "chrf").assign(verdict="❌ incorrecte")])
    other = seg_scores.get(best_direct, pd.DataFrame(columns=["seg_id", "hyp"])).set_index("seg_id").hyp
    rows = "".join(f"<tr><td>{r.verdict}<br>{r.chrf:.0f}</td><td dir='rtl'>{r.transcription}</td><td>{r.ref}</td>"
                   f"<td>{r.hyp}</td><td>{other.get(r.seg_id, '')}</td></tr>" for r in pick.itertuples())
    display(HTML(f"<table><tr><th>verdict / chrF</th><th>source</th><th>référence</th><th>pipeline ({best_casc})</th>"
                 f"<th>direct ({best_direct})</th></tr>{rows}</table>"))

## 10. Traduction vers le français et l'arabe standard
TEDxTN ne fournit que des références anglaises. Pour le français et l'arabe standard (MSA), on compare :
NLLB (`aeb_Arab → fra_Latn / arb_Arab`), le LLM, et le pivot anglais (direct E2E → NLLB `eng_Latn → cible`).
**Évaluation indirecte** : traduction retour vers l'anglais (NLLB) puis chrF++ contre la référence anglaise.
C'est une approximation (la traduction retour ajoute ses propres erreurs), complétée par une lecture manuelle.

In [ ]:
if RUN_OTHER_TARGETS and asr_test is not None:
    sample = test_df.sample(min(150, len(test_df)), random_state=SEED).reset_index(drop=True)
    src = asr_text(sample, asr_test)
    mt_dir = MODELS_DIR / "mt_best" if (MODELS_DIR / "mt_best").exists() else NLLB
    tok, mt = T.load_nllb(str(mt_dir))
    back_tok, back = (tok, mt)
    out_rows = []
    for lang in ["Français", "Arabe standard (MSA)"]:
        code = TARGET_LANGUAGES[lang]["nllb"]
        hyp = T.nllb_translate(tok, mt, src, code)
        back_tok.src_lang = code
        back_en = T.nllb_translate(back_tok, back, hyp, "eng_Latn")
        back_tok.src_lang = "aeb_Arab"
        out_rows.append({"cible": lang, "méthode": "pipeline ASR → NLLB",
                         "chrF++ aller-retour (vs réf. anglaise)": metrics.chrf_fn()(back_en, sample.translation.tolist())})
        sample[f"{lang} (NLLB)"] = hyp
    display(pd.DataFrame(out_rows).round(1))
    display(sample[["transcription", "translation", "Français (NLLB)", "Arabe standard (MSA) (NLLB)"]].head(8))
    sample.to_csv(RESULTS_DIR / "02_other_targets_sample.csv", index=False)
    del mt; torch.cuda.empty_cache()

## 11. Export des modèles pour l'interface

In [ ]:
import shutil
if len(ast_res):
    best_e2e = best_on_dev("direct", prefix="E2E")   # choix sur le dev
    if best_e2e:
        tag = "from-asr" if "from-asr" in best_e2e else ""
        src_dir = MODELS_DIR / f"ast_{E2E_MODEL}-e2e{'-from-asr' if tag else ''}" / "final"
        if src_dir.exists():
            shutil.copytree(src_dir, MODELS_DIR / "ast_best", dirs_exist_ok=True)
    for cand in ["mt_nllb-ft-asrnoise", "mt_nllb-ft"]:
        if (MODELS_DIR / cand / "final").exists():
            shutil.copytree(MODELS_DIR / cand / "final", MODELS_DIR / "mt_best", dirs_exist_ok=True)
            break
    print(sorted(p.name for p in MODELS_DIR.iterdir()))

## 12. Discussion critique et limites

**Direct ou pipeline ?** La réponse dépend des chiffres obtenus en section 8 ; les facteurs à discuter :
- le pipeline bénéficie de NLLB, pré-entraîné sur des milliards de phrases et doté d'un code tunisien ;
  l'approche directe doit tout apprendre à partir de ~22 h de paires audio-anglais ;
- en revanche, le pipeline propage les erreurs ASR (section 9.1) et les erreurs de script des emprunts ;
- l'écart entre C0 (oracle) et C1/C2 mesure exactement le coût de l'ASR imparfait.

**Limites**
- Une seule référence par segment : BLEU sous-estime les traductions valides mais formulées autrement.
  chrF++ et COMET atténuent ce biais sans le supprimer.
- COMET-22 n'a pas été entraîné sur du tunisien : on l'utilise avec la source tunisienne, ce qui reste
  approximatif ; on s'appuie surtout sur les comparaisons relatives.
- Français et arabe standard : aucune référence, évaluation par aller-retour seulement.
- Le LLM est utilisé en 4 bits et en zero/few-shot : un fine-tuning (LoRA) sur les paires TEDxTN
  l'améliorerait probablement.
- Ressources : Whisper-large-v3 et NLLB-1.3B n'ont pas pu être entraînés en complet sur un T4.

**Améliorations envisagées**
- Entraînement **multi-tâche** ASR + AST dans un seul Whisper (tokens de tâche), qui combine les deux signaux.
- Rescoring des n-best du pipeline par COMET-QE ou par le LLM.
- Fine-tuning LoRA du LLM sur les paires TEDxTN ; données parallèles tunisien-anglais supplémentaires.

## 13. Conclusion

In [ ]:
if len(ast_res):
    bd = ast_res[ast_res["système"] == best_on_dev("direct")]; bc = ast_res[ast_res["système"] == best_on_dev("pipeline")]
    for lbl, b in [("direct (choisi sur le dev)", bd), ("pipeline (choisi sur le dev)", bc)]:
        if len(b):
            r = b.iloc[0]
            print(f"Meilleur {lbl} : {r['système']} — BLEU {r['BLEU']:.2f} {r['IC95 BLEU']}, chrF++ {r['chrF++']:.2f}, COMET {r['COMET']}")
    print("Référence papier (direct, Whisper-large-v3 fine-tuné) : 25,68 BLEU.")

## Références
- F. Bougares et al. *TEDxTN*, ArabicNLP 2025.
- NLLB Team. *No Language Left Behind: Scaling Human-Centered Machine Translation*, 2022.
- Seamless Communication. *SeamlessM4T: Massively Multilingual & Multimodal Machine Translation*, 2023.
- A. Radford et al. *Whisper*, 2022. Y. Liu et al. *mBART*, 2020 ; Y. Tang et al. *mBART-50*, 2020.
- M. Post. *A Call for Clarity in Reporting BLEU Scores* (sacreBLEU), WMT 2018.
- M. Popović. *chrF++: words helping character n-grams*, WMT 2017.
- R. Rei et al. *COMET-22*, WMT 2022.
- A. Bérard et al. ; M. Sperber, M. Paulik. *Speech Translation and the End-to-End Promise*, ACL 2020.
- Qwen Team. *Qwen2.5 Technical Report*, 2024.